In [1]:
import chephy_model as cpm
import gnn_utils
import numpy as np
import torch
import pandas as pd


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /home/dsi/chenbis/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /home/dsi/chenbis/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt


In [2]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

healthy_raw_path = ["/dsi/sbm/chen2/downsampled_data/healthy/TRB_h_v1/", "/dsi/sbm/chen2/downsampled_data/healthy/TRB_h_v2/"]
sick_raw_path = "/dsi/sbm/chen2/downsampled_data/sick/mixcr_4.7_TRB/"

healthy_processed_path = "/dsi/sbm/chen2/processed_data/healthy"
sick_processed_path = "/dsi/sbm/chen2/processed_data/sick"


graph_path = "/dsi/sbm/chen2/graph_data"

In [3]:
from pathlib import Path
file_names_sick = [sick_raw_path + f.name for f in Path(sick_raw_path).iterdir() if f.is_file()]
file_names_healthy = [healthy_raw_path[0] + f.name for f in Path(healthy_raw_path[0]).iterdir() if f.is_file()]
file_names_healthy += [healthy_raw_path[1] + f.name for f in Path(healthy_raw_path[1]).iterdir() if f.is_file()]

In [4]:
def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered

In [5]:
from scipy.spatial.distance import squareform 


def get_graph_data(data, cdr3_header, distance_threshold = 0.2, use_mst=True, enrich=False):

    # data = data.sample(n=n, random_state=42)

    data = prepare_data(data, cdr3_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    # epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    # data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    # print("Done indexing")

    graph = gnn_utils.build_graph(
                            # data,
                            squareform(chephy_matrix),
                            sequences_list,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    return graph, data


In [6]:
from tqdm import tqdm
import os

cdr3_header = "aaSeqCDR3"

distance_threshold=0.05
use_mst=False
enrich=True

sub_folder = f"{graph_path}/{distance_threshold}_{use_mst}_{enrich}"

if not os.path.exists(sub_folder):
    os.makedirs(sub_folder)

for file_name in tqdm(file_names_sick, desc="Processing sick files"):
    df = pd.read_csv(file_name, sep="\t")
    data = get_graph_data(df, cdr3_header, distance_threshold=distance_threshold, use_mst=use_mst, enrich=enrich)

    torch.save(data, f"{sub_folder}/sick_{file_name.rsplit('/', 1)[-1]}.pt")

for file_name in tqdm(file_names_healthy, desc="Processing healthy files"):
    df = pd.read_csv(file_name, sep="\t")
    data = get_graph_data(df, cdr3_header, distance_threshold=distance_threshold, use_mst=use_mst, enrich=enrich)

    torch.save(data, f"{sub_folder}/healthy_{file_name.rsplit('/', 1)[-1]}.pt")

Processing healthy files: 100%|██████████| 51/51 [1:49:17<00:00, 128.58s/it]


In [2]:
import os
import torch
from tqdm import tqdm

folder_path = "/dsi/sbm/chen2/graph_data/0.05_False_True"

# Get all .pt files
pt_files = [f for f in os.listdir(folder_path) if f.endswith(".pt")]

for filename in pt_files:

    prefix = filename.split('_', 1)[0]  # Get part before first underscore

    file_path = os.path.join(folder_path, filename)
    graph, df = torch.load(file_path, weights_only=False)
    print(graph.x.shape, graph.edge_index.shape)
    # print(graph.y)
    # graph.y = torch.tensor(0.0) if prefix == "healthy" else torch.tensor(1.0)
    # torch.save((graph, df), file_path)
    
    

torch.Size([11827, 320]) torch.Size([2, 92904])
torch.Size([9043, 320]) torch.Size([2, 49376])
torch.Size([27936, 320]) torch.Size([2, 497108])
torch.Size([23264, 320]) torch.Size([2, 345286])
torch.Size([12410, 320]) torch.Size([2, 109070])
torch.Size([14253, 320]) torch.Size([2, 129712])
torch.Size([28776, 320]) torch.Size([2, 518480])
torch.Size([29826, 320]) torch.Size([2, 587414])
torch.Size([14357, 320]) torch.Size([2, 142380])
torch.Size([28272, 320]) torch.Size([2, 558402])
torch.Size([29821, 320]) torch.Size([2, 574310])
torch.Size([14218, 320]) torch.Size([2, 132538])
torch.Size([34919, 320]) torch.Size([2, 757512])
torch.Size([34678, 320]) torch.Size([2, 801024])
torch.Size([31979, 320]) torch.Size([2, 649836])
torch.Size([18130, 320]) torch.Size([2, 205726])
torch.Size([12886, 320]) torch.Size([2, 107484])
torch.Size([28694, 320]) torch.Size([2, 549982])
torch.Size([6030, 320]) torch.Size([2, 25352])
torch.Size([13718, 320]) torch.Size([2, 115758])
torch.Size([31118, 320]) 

In [8]:
graph

Data(x=[32393, 320], edge_index=[2, 689258], y=0.0)